## Imports

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
sample_submission = pd.read_csv('../data/sample_submission.csv')

TARGET = 'addicted_label'
ID_COL = 'id'

print(train.shape, test.shape, sample_submission.shape)

(691369, 14) (296302, 13) (296302, 2)


## Prepare categorical features

In [2]:
categorical_features = ["gender", "stress_level", "academic_work_impact"]

for col in categorical_features:
    train[col] = train[col].fillna("Missing").astype(object).astype(str)
    test[col] = test[col].fillna("Missing").astype(object).astype(str)

    combined = pd.concat([train[col], test[col]], axis=0)
    categories = combined.astype("category").cat.categories

    train[col] = pd.Categorical(train[col], categories=categories).codes.astype("int32")
    test[col] = pd.Categorical(test[col], categories=categories).codes.astype("int32")

print(train[categorical_features].dtypes)
print(test[categorical_features].dtypes)

gender                  int32
stress_level            int32
academic_work_impact    int32
dtype: object
gender                  int32
stress_level            int32
academic_work_impact    int32
dtype: object


## Feature Engineering

In [3]:
for df in [train, test]:
    df["total_screen_time"] = (
        df["daily_screen_time_hours"].fillna(0)
        + df["weekend_screen_time"].fillna(0)
        + df["social_media_hours"].fillna(0)
    )
    df["weekend_vs_weekday_ratio"] = (
        df["weekend_screen_time"] / df["daily_screen_time_hours"].replace(0, np.nan)
    )

print(train[["total_screen_time", "weekend_vs_weekday_ratio"]].describe())

       total_screen_time  weekend_vs_weekday_ratio
count      691369.000000             517906.000000
mean           16.516919                  1.330336
std             7.522305                  0.498424
min             0.000000                  0.085427
25%            11.000000                  1.075426
50%            16.250000                  1.240741
75%            22.890000                  1.451613
max            37.110000                 20.440000


## Build X/y

In [4]:
X = train.drop(columns=[TARGET, ID_COL])
y = train[TARGET]
X_test = test.drop(columns=[ID_COL])

print(X.dtypes)

age                         float64
daily_screen_time_hours     float64
social_media_hours          float64
gaming_hours                float64
work_study_hours            float64
sleep_hours                 float64
notifications_per_day       float64
app_opens_per_day           float64
weekend_screen_time         float64
gender                        int32
stress_level                  int32
academic_work_impact          int32
total_screen_time           float64
weekend_vs_weekday_ratio    float64
dtype: object


## LightGBM seed-averaged ensemble

In [5]:
seeds = [42, 1, 7, 2024, 99]
lgb_test_preds_list = []
lgb_val_scores = []

for seed in seeds:
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=seed
    )

    seed_model = lgb.LGBMClassifier(
        n_estimators=3000,
        learning_rate=0.02,
        num_leaves=64,
        random_state=seed,
        n_jobs=-1
    )

    seed_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
    )

    val_preds = seed_model.predict_proba(X_val)[:, 1]
    val_auc = roc_auc_score(y_val, val_preds)
    lgb_val_scores.append(val_auc)
    print(f"[LGBM] Seed {seed}: Validation AUC = {val_auc:.5f}")

    final_model = lgb.LGBMClassifier(
        n_estimators=seed_model.best_iteration_,
        learning_rate=0.02,
        num_leaves=64,
        random_state=seed,
        n_jobs=-1
    )
    final_model.fit(X, y)
    lgb_test_preds_list.append(final_model.predict_proba(X_test)[:, 1])

print(f"\nMean LGBM validation AUC: {np.mean(lgb_val_scores):.5f}")
lgb_test_preds = np.mean(lgb_test_preds_list, axis=0)

/home/nesrine/miniconda3/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 392379, number of negative: 160716
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005924 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2467
[LightGBM] [Info] Number of data points in the train set: 553095, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.709424 -> initscore=0.892589
[LightGBM] [Info] Start training from score 0.892589
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2999]	valid_0's auc: 0.962415	valid_0's binary_logloss: 0.224227
[LGBM] Seed 42: Validation AUC = 0.96241
[LightGBM] [Info] Number of positive: 490474, number of negative: 200895
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004353 seconds.
You can set `force_row_wise=true` to remove the

/home/nesrine/miniconda3/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 392379, number of negative: 160716
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005319 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2467
[LightGBM] [Info] Number of data points in the train set: 553095, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.709424 -> initscore=0.892589
[LightGBM] [Info] Start training from score 0.892589
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2993]	valid_0's auc: 0.963287	valid_0's binary_logloss: 0.22179
[LGBM] Seed 1: Validation AUC = 0.96329
[LightGBM] [Info] Number of positive: 490474, number of negative: 200895
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005492 seconds.
You can set `force_row_wise=true` to remove the o

/home/nesrine/miniconda3/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 392379, number of negative: 160716
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004120 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2467
[LightGBM] [Info] Number of data points in the train set: 553095, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.709424 -> initscore=0.892589
[LightGBM] [Info] Start training from score 0.892589
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2999]	valid_0's auc: 0.96234	valid_0's binary_logloss: 0.224331
[LGBM] Seed 7: Validation AUC = 0.96234
[LightGBM] [Info] Number of positive: 490474, number of negative: 200895
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006048 seconds.
You can set `force_row_wise=true` to remove the o

/home/nesrine/miniconda3/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 392379, number of negative: 160716
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006599 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2466
[LightGBM] [Info] Number of data points in the train set: 553095, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.709424 -> initscore=0.892589
[LightGBM] [Info] Start training from score 0.892589
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2994]	valid_0's auc: 0.962799	valid_0's binary_logloss: 0.223333
[LGBM] Seed 2024: Validation AUC = 0.96280
[LightGBM] [Info] Number of positive: 490474, number of negative: 200895
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005930 seconds.
You can set `force_row_wise=true` to remove t

/home/nesrine/miniconda3/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 392379, number of negative: 160716
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004935 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2467
[LightGBM] [Info] Number of data points in the train set: 553095, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.709424 -> initscore=0.892589
[LightGBM] [Info] Start training from score 0.892589
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2993]	valid_0's auc: 0.963073	valid_0's binary_logloss: 0.222418
[LGBM] Seed 99: Validation AUC = 0.96307
[LightGBM] [Info] Number of positive: 490474, number of negative: 200895
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005145 seconds.
You can set `force_row_wise=true` to remove the

## CatBoost seed-averaged ensemble

In [6]:
cat_test_preds_list = []
cat_val_scores = []

for seed in seeds:
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=seed
    )

    cat_model = CatBoostClassifier(
        iterations=3000,
        learning_rate=0.03,
        depth=8,
        eval_metric="AUC",
        random_seed=seed,
        verbose=0,
        early_stopping_rounds=100
    )

    cat_model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        use_best_model=True
    )

    val_preds = cat_model.predict_proba(X_val)[:, 1]
    val_auc = roc_auc_score(y_val, val_preds)
    cat_val_scores.append(val_auc)
    print(f"[CatBoost] Seed {seed}: Validation AUC = {val_auc:.5f}")

    final_cat = CatBoostClassifier(
        iterations=cat_model.get_best_iteration(),
        learning_rate=0.03,
        depth=8,
        random_seed=seed,
        verbose=0
    )
    final_cat.fit(X, y)
    cat_test_preds_list.append(final_cat.predict_proba(X_test)[:, 1])

print(f"\nMean CatBoost validation AUC: {np.mean(cat_val_scores):.5f}")
cat_test_preds = np.mean(cat_test_preds_list, axis=0)

[CatBoost] Seed 42: Validation AUC = 0.96168
[CatBoost] Seed 1: Validation AUC = 0.96258
[CatBoost] Seed 7: Validation AUC = 0.96183
[CatBoost] Seed 2024: Validation AUC = 0.96217
[CatBoost] Seed 99: Validation AUC = 0.96212

Mean CatBoost validation AUC: 0.96208


## Blend Catboost and LightGBM

In [7]:
final_test_preds = 0.5 * lgb_test_preds + 0.5 * cat_test_preds

## Submission

In [ ]:
submission = sample_submission.copy()
submission[TARGET] = final_test_preds
submission.to_csv("../submissions/submission_blend.csv", index=False)
submission.head()

,id,addicted_label
0,691369,0.999196
1,691370,0.945835
2,691371,0.949396
3,691372,0.987531
4,691373,0.997351


In [9]:
# Try weighting LightGBM more heavily since it's the stronger model
final_test_preds2 = 0.75 * lgb_test_preds + 0.25 * cat_test_preds

In [10]:
submission = sample_submission.copy()
submission[TARGET] = final_test_preds
submission.to_csv("../submissions/submission_blend_weighted.csv", index=False)
submission.head()

,id,addicted_label
0,691369,0.999196
1,691370,0.945835
2,691371,0.949396
3,691372,0.987531
4,691373,0.997351


In [11]:
submission = sample_submission.copy()
submission[TARGET] = lgb_test_preds
submission.to_csv("submission_lgb_features_only.csv", index=False)
submission.head()

,id,addicted_label
0,691369,0.998976
1,691370,0.966173
2,691371,0.942552
3,691372,0.987244
4,691373,0.997279
